In [1]:
import os
import numpy as np
import ROOT

Welcome to JupyROOT 6.30/04


In [2]:
# Asegurarse que AUGEROFFLINEROOT está definido
AugerOfflineRoot = os.environ.get("AUGEROFFLINEROOT")
if AugerOfflineRoot is None:
    raise EnvironmentError("AUGEROFFLINEROOT no definido. Hacé 'source this-auger-offline.sh' primero.")

# -----------------------------
# 1️⃣ Cargar librerías Offline
# -----------------------------
lib_dir = os.path.join(AugerOfflineRoot, "lib")

libs_to_load = [
    "libRecEventKG.so"
]

for lib in libs_to_load:
    full_path = os.path.join(lib_dir, lib)
    if not os.path.exists(full_path):
        print(f"[WARN] Librería {lib} no encontrada en {lib_dir}")
        continue
    ROOT.gSystem.Load(full_path)

In [3]:

# -----------------------------
# 2️⃣ Funciones auxiliares
# -----------------------------
def getCounterList(sevent, mevent):
    cList = []
    stationVector = sevent.GetStationVector()
    for station in stationVector:
        stationId = station.GetId()
        counterId = int("10" + str(stationId))
        if mevent.HasCounter(counterId):
            cList.append(mevent.GetCounter(counterId))
    return cList

def getModuleList(counter, sim=True):
    possibleModules = range(0,6) if sim else range(100,116)
    modules = []
    for modId in possibleModules:
        if counter.HasModule(modId):
            modules.append(counter.GetModule(modId))
    return modules

# -----------------------------
# 3️⃣ Función principal
# -----------------------------
def readADST_sim(fname):
    # Preparar ADST
    DataFiles = fname if isinstance(fname, list) else [fname]
    files = ROOT.std.vector('string')()
    for datfile in DataFiles:
        files.push_back(datfile)

    # Crear RecEventFile y RecEvent
    file1 = ROOT.RecEventFile(files)
    event = ROOT.RecEvent()
    geo = ROOT.DetectorGeometry()
    file1.ReadDetectorGeometry(geo)
    file1.SetBuffers(event)

    nMod = 0

    # Loop sobre eventos
    while file1.ReadNextEvent() == ROOT.RecEventFile.eSuccess:
        MCShower = event.GetGenShower()
        ee = MCShower.GetEnergy()
        thetaMC = MCShower.GetZenith() * 180.0/np.pi
        phiMC = MCShower.GetAzimuth() * 180.0/np.pi
        pp = MCShower.GetShortPrimaryName()

        sEvent = event.GetSDEvent()
        sShower = sEvent.GetSdRecShower()
        zenithAngle = sShower.GetZenith() * 180.0/np.pi
        energy = sShower.GetEnergy()
        azimuthAngle = sShower.GetAzimuth() * 180.0/np.pi

        mEvent = event.GetMDEvent()
        counterList = getCounterList(sEvent, mEvent)

        for counter in counterList:
            if counter.IsRejected() or counter.IsSaturated():
                continue
            modulesList = getModuleList(counter)
            for mod in modulesList:
                if mod.IsRejected() or mod.IsSaturated():
                    continue
                nMod += 1

    return nMod


In [4]:
# Ruta a tu ADST de simulación
fname = "ADST_SUPERAMIGA_230cm_Proton_e1850_t15_v2.root"

nModulos = readADST_sim(fname)
print(f"Número de módulos procesados: {nModulos}")

Número de módulos procesados: 945


  input file ADST_SUPERAMIGA_230cm_Proton_e1850_t15_v2.root
Warning in <TClass::Init>: no dictionary for class RdBeamQuants is available
Warning in <TClass::Init>: no dictionary for class RdBeamPeak is available
Warning in <TClass::Init>: no dictionary for class SimRdPulse is available
Warning in <TStreamerInfo::BuildOld>: Cannot convert Detector::fTelPointingIds from type: map<int,EyePointingIdMap> to type: map<int,map<int,TString> >, skip element
Warning in <TStreamerInfo::BuildOld>: Cannot convert TFormula::fParams from type: Double_t* to type: map<TString,int,TFormulaParamOrder>, skip element
Warning in <TStreamerInfo::BuildOld>: Cannot convert TFormula::fLinearParts from type: TObjArray to type: vector<TObject*>, skip element
Warning in <TStreamerInfo::Compile>: Counter fNpar should not be skipped from class TFormula
Warning in <TStreamerInfo::Compile>: Counter fNoper should not be skipped from class TFormula
Warning in <TStreamerInfo::Compile>: Counter fNconst should not be skipp